# P1 – Fusion-LR (Sơn)

Same logistic regression as the baselines (`fusion_lr`): P, C and G standardized with train statistics, T raw,
`class_weight="balanced"`, C in {0.01, 0.1, 1, 10} chosen on validation PR-AUC, threshold from `best_threshold`.

- **P1 (P + C + T + G):** content, the 4 profile counts and the 28 computed graph features.
- **P1 + G-emb (P + C + T + G + N2V):** adds the 128-d Node2Vec embedding (B2b settings: p = 1, q = 0.5, 20 epochs).
- **P + C + T + N2V:** Node2Vec in place of G, to see how much the two overlap.

Validation only; the test set is not used.

In [1]:
# 1. Setup (Colab or local)
import os, sys
if not os.path.exists("mgtab.py"):
    if "google.colab" in sys.modules:
        if not os.path.exists("sna-content-moderation"):
            !git clone -q https://github.com/ThSonDev/sna-content-moderation.git
        %cd sna-content-moderation
        !pip install -q -r requirements.txt
    else:
        %cd ..
sys.path.insert(0, os.getcwd())

/mnt/DevZone/social-networks/project/sna-content-moderation


In [2]:
# 2. Download MGTAB (31 MB) into data/MGTAB, only once
import urllib.request, zipfile
if not os.path.exists("data/MGTAB/features.pt"):
    os.makedirs("data", exist_ok=True)
    url = "https://drive.usercontent.google.com/download?id=1gbWNOoU1JB8RrTu2a5j9KMNVa9wX72Fe&export=download&confirm=t"
    urllib.request.urlretrieve(url, "data/MGTAB.zip")
    zipfile.ZipFile("data/MGTAB.zip").extractall("data")

In [3]:
# 3. Load data and the shared split (never make your own split)
import numpy as np
from mgtab import load, relations, split, best_threshold, evaluate, RELATIONS
X, y, stance, E, T, W = load()
rel = relations(E, T, len(y))   # {"follow": (src, dst), ...}
train, val, test = split(y)     # test stays locked until the final run
print(X.shape, {r: len(s) for r, (s, d) in rel.items()})

(10199, 788) {'follow': 425863, 'mention': 110868, 'reply': 22327, 'quote': 20901, 'url': 263800, 'hashtag': 300000}


## Features

In [4]:
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from features import feature_groups
from embeddings import merged_edge_index, train_node2vec

groups, g_names = feature_groups(X, rel)   # P 16, C 4, T 768, G 28; G cached in data/g_struct.npz

# Node2Vec embedding from baselines.ipynb (B2b). Derived from MGTAB, so it lives in data/ and is never pushed.
N2V = Path("data/node2vec/node2vec_p1.0_q0.5_e20.npy")
if not N2V.exists():   # about 70 minutes on Colab CPU; get the file from the team Drive instead if you can
    Z_n2v, _ = train_node2vec(merged_edge_index(rel, len(y)), len(y), 1.0, 0.5, output_dir=str(N2V.parent / "e20"),
                              epochs=20, batch_size=256, lr=0.01, seed=42)
    N2V.parent.mkdir(parents=True, exist_ok=True); np.save(N2V, Z_n2v)
groups["N2V"] = np.load(N2V)

Z = {k: StandardScaler().fit(v[train]).transform(v) for k, v in groups.items() if k != "T"}  # train statistics only
Z["T"] = groups["T"]                                                                       # T raw, as in the baselines
{k: v.shape for k, v in Z.items()}

/mnt/DevZone/social-networks/project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/mnt/DevZone/social-networks/project/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


{'P': (10199, 16),
 'C': (10199, 4),
 'G': (10199, 28),
 'N2V': (10199, 128),
 'T': (10199, 768)}

## Train and evaluate (validation only)

In [5]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

def fusion_lr(F):
    best = None
    for C in [0.01, 0.1, 1, 10]:
        clf = LogisticRegression(C=C, class_weight="balanced", max_iter=5000).fit(F[train], y[train])
        s = clf.predict_proba(F[val])[:, 1]
        ap = average_precision_score(y[val], s)
        if best is None or ap > best[0]:
            best = (ap, C, s)
    _, C, s = best
    return {"C": C, **evaluate(y[val], s, best_threshold(y[val], s))}, s

models = {"P1 Fusion-LR (P+C+T+G)": ["P", "C", "T", "G"],
          "P1 + G-emb (P+C+T+G+N2V)": ["P", "C", "T", "G", "N2V"],
          "P+C+T+N2V": ["P", "C", "T", "N2V"]}
rows, scores = {}, {}
for name, ks in models.items():
    rows[name], scores[name] = fusion_lr(np.hstack([Z[k] for k in ks]))
res = pd.DataFrame(rows).T
res.round(4)

,C,pr_auc,roc_auc,macro_f1,accuracy,p@100,p@200,threshold
P1 Fusion-LR (P+C+T+G),0.1,0.9153,0.9711,0.9065,0.9235,0.97,0.950,0.6419
P1 + G-emb (P+C+T+G+N2V),0.1,0.9192,0.9695,0.9036,0.9209,0.99,0.950,0.6054
P+C+T+N2V,0.1,0.9005,0.9646,0.8972,0.9170,0.96,0.925,0.6448


## Does Node2Vec add anything?

95% paired bootstrap of the PR-AUC change (1,000 resamples of the validation accounts, same resample for both
models; C and threshold fixed). If the interval contains 0, the change is noise. Same method as `relations.ipynb`.

In [6]:
rng = np.random.default_rng(42)
yv = y[val]
boots = [rng.choice(len(val), len(val)) for _ in range(1000)]

def delta_ci(a, b):
    d = [average_precision_score(yv[i], scores[a][i]) - average_precision_score(yv[i], scores[b][i]) for i in boots]
    return res.loc[a, "pr_auc"] - res.loc[b, "pr_auc"], *np.percentile(d, [2.5, 97.5])

ref = "P1 Fusion-LR (P+C+T+G)"
res["delta_pr_auc_vs_p1"], res["ci_low"], res["ci_high"] = zip(*[delta_ci(m, ref) if m != ref else (0.0, np.nan, np.nan)
                                                                  for m in res.index])
res.to_csv("results/p1.csv")
res[["C", "pr_auc", "delta_pr_auc_vs_p1", "ci_low", "ci_high", "macro_f1", "accuracy", "p@100", "p@200"]].round(4)

,C,pr_auc,delta_pr_auc_vs_p1,ci_low,ci_high,macro_f1,accuracy,p@100,p@200
P1 Fusion-LR (P+C+T+G),0.1,0.9153,0.0000,NaN,NaN,0.9065,0.9235,0.97,0.950
P1 + G-emb (P+C+T+G+N2V),0.1,0.9192,0.0039,-0.0088,0.0195,0.9036,0.9209,0.99,0.950
P+C+T+N2V,0.1,0.9005,-0.0148,-0.0311,0.0019,0.8972,0.9170,0.96,0.925
